In [10]:

from replication import *
import os 
from tool import *
from daily_job import *
from pythonTool import *
from riskModule import *


In [11]:
current_date = '20250915'

# 同步所有策略持仓

In [85]:
get_all_strategy(current_date)


# 头寸调整

In [12]:
adjust_all_strategy(current_date)

  0%|          | 0/3 [00:00<?, ?it/s]

开始优化组合: CSI1000
000852.XSHG
000852.XSHG 20250915 None (20, 10000000)
[Industry exposure(sw_l1) calculated]  0.090 sec
[Log market cap exposure calculated]  0.040 sec


 33%|███▎      | 1/3 [00:04<00:09,  4.76s/it]

开始优化组合: CSI500
000905.XSHG
000905.XSHG 20250915 None (20, 10000000)
[Industry exposure(sw_l1) calculated]  0.072 sec
[Log market cap exposure calculated]  0.014 sec


 67%|██████▋   | 2/3 [00:06<00:02,  2.96s/it]

开始优化组合: CSI300
000300.XSHG
000300.XSHG 20250915 None (20, 10000000)
[Industry exposure(sw_l1) calculated]  0.093 sec
No data for following codes. Codes will be dropped
['601989.XSHG']
[Log market cap exposure calculated]  0.012 sec


100%|██████████| 3/3 [00:07<00:00,  2.54s/it]

[组合策略目标持仓优化完成]  7.619 sec


In [ ]:
from pythonTool import *
from riskModule import *

In [5]:
market_caps = ddb(db='percy').load_table_by_sql('dfs://jq_fundamental', 'valuation', f'select code, market_cap from valuation where date=2025.09.01')
market_caps

""


In [4]:
get_previous_tradedate('2025-09-01', 3)

IndexError: single positional indexer is out-of-bounds

In [14]:
sell_df = pd.read_excel('./Trading/{current_date}/component_change.xlsx'.format(current_date=current_date), sheet_name='sell', dtype={'ticker':str})

In [22]:
valuation_df = ddb().load_table_by_sql('dfs://jq_fundamental', 'valuation', 'select * from valuation where date=2025.05.02')
price_df = ddb().load_table_by_sql('dfs://jq_price', 'daily', 'select * from daily where date=2025.05.26')


In [36]:
valuation_df['ticker'] = valuation_df['code'].apply(lambda x: x[:6])
market_cap_mapping = valuation_df.set_index('ticker')['market_cap'].to_dict()
price_df['ticker'] = price_df['code'].apply(lambda x: x[:6])
close_mapping = price_df.set_index('ticker')['close'].to_dict()
sell_df['market_cap'] = sell_df['ticker'].map(market_cap_mapping)
sell_df['close'] = sell_df['ticker'].map(close_mapping)
sell_df['market_value'] = sell_df['total pos'] * sell_df['close']
total_market_value = sell_df['market_value'].sum()

print('总计需要卖出:',total_market_value)

总计需要卖出: 479410.0


In [47]:
buy_df = pd.read_excel('./Trading/{current_date}/component_change.xlsx'.format(current_date=current_date), sheet_name='buy', dtype={'ticker':str})
buy_df['market_cap'] = buy_df['ticker'].map(market_cap_mapping)
total_market_cap = buy_df['market_cap'].sum()
buy_df['ratio'] = buy_df['market_cap'] / total_market_cap
buy_df['target_market_value'] = buy_df['ratio'] * total_market_value
buy_df['close'] = buy_df['ticker'].map(close_mapping)
buy_df['target_pos'] = buy_df['target_market_value'] / buy_df['close']
buy_df

,ticker,market_cap,ratio,target_market_value,close,target_pos
0,000589,67.8091,0.157957,75725.962622,4.48,16903.116657
1,301215,73.3932,0.170964,81962.018666,5.98,13706.023188
2,301377,108.7730,0.253379,121472.488682,27.30,4449.541710
3,601089,75.3600,0.175546,84158.446922,16.13,5217.510659
4,603227,103.9542,0.242154,116091.083108,9.13,12715.343166


In [52]:
wb = pd.ExcelWriter('调仓.xlsx')
buy_df.to_excel(wb, sheet_name='buy', index='ticker')
sell_df.to_excel(wb, sheet_name='sell', index='ticker')
wb.close()

In [3]:
jq_ticker_mapping = {'SH000905 Index': '000905.XSHG',
                      'SHSZ300 Index': '000300.XSHG',
                      'CSI1000 Index': '000852.XSHG'}

strategy_config = pd.read_excel(project_path + 'Script/组合管理.xlsx', sheet_name='组合管理')

total_deviation = []
industry_deviation = []
log_market_cap_deviation = []
strategy_stock_num = []
make_path(project_path + f'Output/{current_date}/')
make_path(project_path + f'Output/{current_date}/策略目标仓位/')
make_path(project_path + f'Output/{current_date}/投资组合当日流水/')
make_path(project_path + f'Output/{current_date}/投资组合交易指令/')

strategy_weight = {}

# 优化计算策略目标仓位
with timer('组合策略目标持仓优化完成'):
    for i in tqdm.trange(len(strategy_config)):
        strategy_info = strategy_config.iloc[i]
        strategy_name = strategy_info['策略编号']
        print(f'开始优化组合: {strategy_name}')
        index_jq_ticker = jq_ticker_mapping.get(strategy_info['挂钩指数'], strategy_info['挂钩指数'])

        print(index_jq_ticker)

        weight_df, strategy_exposures = replicate_index(index=index_jq_ticker, date=current_date, 
                                                    exclude_restricted=strategy_info['禁投池限制'], 
                                                    exclude_st=strategy_info['ST限制'], 
                                                    exclude_paused=strategy_info['停牌限制'],
                                                    industry_deviation=strategy_info['行业偏离度'], 
                                                    log_market_cap_deviation=strategy_info['对数市值偏离度'],
                                                    min_avg_amount=(strategy_info['滚动天数'], strategy_info['最低成交额限制']), 
                                                    min_price=None,
                                                    notional=strategy_info['组合规模'])
        
        # 输出目标仓位
        weight_df.to_excel(project_path + f'Output/{current_date}/策略目标仓位/{strategy_name}目标仓位.xlsx')
        # 生成交易文件
        weight_df['code2'] = weight_df.index
        weight_df['市场'] = weight_df['code2'].apply(lambda x: x[-1])
        weight_df['市场'] = weight_df['市场'].apply(lambda x: 1 if x=='E' else 0)
        weight_df['代码'] = weight_df['code2'].apply(lambda x: x[:6])
        weight_df['委托数量'] = weight_df['组合实际数量']
        weight_df['买卖方向'] = 'B'
        weight_df['产品'] = 2
        weight_df['资产单元'] = 139
        weight_df['投资组合'] = 144
        weight_df['名称'] = None
        weight_df['开平标志'] = None
        weight_df['投保标志'] = None
        weight_df['委托价格'] = None
        trade_df = weight_df.loc[weight_df['委托数量'] > 0]
        trade_df = trade_df[['产品','资产单元',	'投资组合',	'市场',	'代码',	'名称',	'买卖方向',	'开平标志',	'投保标志',	'委托价格',	'委托数量']]
        trade_df.to_excel(project_path + f'Output/{current_date}/策略目标仓位/{strategy_name} IMS交易指令模板.xlsx', index=False)

        strategy_weight[strategy_name] = weight_df

        total_deviation.append(strategy_exposures['total deviation'])
        industry_deviation.append(strategy_exposures['industry deviation'])
        log_market_cap_deviation.append(strategy_exposures['log_market_cap deviation'])
        strategy_stock_num.append(len(weight_df.loc[weight_df['组合实际数量'] > 0]))


  0%|          | 0/3 [00:00<?, ?it/s]

开始优化组合: CSI1000
000852.XSHG


  0%|          | 0/3 [00:00<?, ?it/s]


KeyError: "None of ['is_st'] are in the columns"

In [4]:
st_info = ddb().load_table_by_sql('dfs://jq_price', 'st', f'select code, is_st from st where date={2025.09.01}')
st_list = list(st_info.set_index('is_st')['code'].loc[1])

SyntaxError: f-string: invalid syntax (2277388977.py, line 1)

In [11]:
ddb().load_table_by_sql('dfs://jq_price', 'st', f'select code, is_st from st where date=2025.08.05')

""
